### Limpieza de los espectros

Este código verifica si es que existen gaps o valores extraños de flujo en los espectros cortados con el algoritmo max-min_cut



In [ ]:
import os
from astropy.io import fits
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/drive/')

Drive already mounted at /drive/; to attempt to forcibly remount, call drive.mount("/drive/", force_remount=True).


In [ ]:
#!ls '/drive/My Drive/Data/spec_data/cut_spec'

In [ ]:
dir = '/drive/My Drive/Data/spec_data/cut_spec/'  #path donde estan los espectros

output_directory = '/drive/My Drive/Data/spec_data/cleaned_spec/'  #path donde se guardan los espectros sin gaps
#crear carpeta de salida si no existe
os.makedirs(output_directory, exist_ok=True)

In [ ]:
col_loglam = 'loglam'
loglam_threshold = 0.0002
col_flux = 'flux'
flux_min = -40.0
flux_max = 800.0

In [ ]:
#crear una lista con los espectros a procesar
fits_files = [os.path.join(dir, file) for file in os.listdir(dir)
    if file.startswith('cortado') and file.endswith('.fits')]

In [ ]:
#fits_files

In [ ]:
#procesar los espectros eliminando los que no pasen el filtro
for fits_file in fits_files:
    hdul = fits.open(fits_file)

    if len(hdul) > 1 and hasattr(hdul[1], 'data'):
        data = hdul[1].data

        #guardar los valores de loglam y flux
        loglam_values = data[col_loglam]
        flux_values = data[col_flux]

        #verificar las discontinuidades en loglam
        loglam_diffs = loglam_values[1:] - loglam_values[:-1]
        has_discontinuity = any(loglam_diffs > loglam_threshold)

        #verificar si flux esta fuera del rango permitido
        flux_out_of_range = (flux_values <= flux_min) | (flux_values >= flux_max)

        if has_discontinuity:
            print(f'El espectro {fits_file} tiene una discontinuidad en log lam.')
        elif any(flux_out_of_range):
            print(f'El espectro {fits_file} tiene valores de flux fuera del rango [{flux_min}, {flux_max}].')
        else:
            #guardar el archivo si pasa los filtros

            #cambiar el nombre del archivo
            #quitarle el 'cortado_'
            base_name = os.path.basename(fits_file).replace('cortado_', '')

            #agregar el prefijo "cleaned_"
            output_file = os.path.join(output_directory, f'cleaned_{base_name}')

            #borrar el archivo anterior si ya existe
            if os.path.exists(output_file):
              os.remove(output_file)

            hdul.writeto(output_file, overwrite=True)
            #print(f"El espectro {fits_file} pasa los filtros y se guardó como {output_file}.")

    hdul.close()



El espectro /drive/My Drive/Data/spec_data/cut_spec/cortado_spec-0303-51615-0371.fits tiene valores de flux fuera del rango [-40.0, 800.0].
El espectro /drive/My Drive/Data/spec_data/cut_spec/cortado_spec-0303-51615-0632.fits tiene valores de flux fuera del rango [-40.0, 800.0].
El espectro /drive/My Drive/Data/spec_data/cut_spec/cortado_spec-0303-51615-0640.fits tiene valores de flux fuera del rango [-40.0, 800.0].
El espectro /drive/My Drive/Data/spec_data/cut_spec/cortado_spec-0305-51613-0594.fits tiene valores de flux fuera del rango [-40.0, 800.0].
El espectro /drive/My Drive/Data/spec_data/cut_spec/cortado_spec-0309-51994-0055.fits tiene valores de flux fuera del rango [-40.0, 800.0].
El espectro /drive/My Drive/Data/spec_data/cut_spec/cortado_spec-0312-51689-0508.fits tiene valores de flux fuera del rango [-40.0, 800.0].
El espectro /drive/My Drive/Data/spec_data/cut_spec/cortado_spec-0326-52375-0607.fits tiene valores de flux fuera del rango [-40.0, 800.0].


In [ ]:
cleaned_files = len(os.listdir(output_directory))
print(f'Hay {cleaned_files} archivos en la carpeta {output_directory}')

Hay 121 archivos en la carpeta /drive/My Drive/Data/spec_data/cleaned_spec/
